# Lecture 7: Turn the Fire Model into a Flask Web App

This lesson converts the Algerian Forest Fires Ridge model into a small web prediction service. A user enters weather values in a form; Flask sends them to the saved model and returns a predicted FWI.

![Flask framework logo](https://images.microcms-assets.io/assets/18598a9a51944b7daa5722b0fd747fee/64cdd871d4784ff9bc9b4a4c6aeb2794/flask_logo.png)

Image source: [Flask logo](https://www.kikagaku.co.jp/personal/blog/flask). Flask template and request behaviour: [official Flask documentation](https://flask.palletsprojects.com/en/stable/quickstart/).

## Machine-learning project life cycle

Data collection → EDA → feature engineering → feature selection → model training → save model → web app → cloud deployment.

The earlier lessons completed the data and model stages. This notebook focuses on the web-app stage. Deployment to AWS is the next separate step.

In [ ]:
# Cell 1: draw the end-to-end request flow
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

steps = ['Browser form', 'Flask route', 'Validate inputs', 'Ridge pipeline', 'FWI result']
colors = ['#DCEAF7', '#CDEFD9', '#FFF1C9', '#F8D5D5', '#E6D8F5']
fig, ax = plt.subplots(figsize=(14, 3))
ax.set_xlim(0, 14)
ax.set_ylim(0, 3)
ax.axis('off')
for index, (step, color) in enumerate(zip(steps, colors)):
    x = 0.3 + index * 2.75
    box = FancyBboxPatch((x, 1), 2.1, 0.75, boxstyle='round,pad=0.08', facecolor=color, edgecolor='#444')
    ax.add_patch(box)
    ax.text(x + 1.05, 1.38, step, ha='center', va='center', fontsize=11, weight='bold')
    if index < len(steps) - 1:
        ax.annotate('', xy=(x + 2.65, 1.38), xytext=(x + 2.15, 1.38), arrowprops={'arrowstyle': '->', 'lw': 2})
ax.set_title('What happens after the user presses Predict', fontsize=15, weight='bold')
plt.show()

## Project structure

A small Flask project is easier to understand when each type of file has one home:

- application.py — Flask routes and prediction logic
- models/ridge_pipeline.pkl — the saved fitted model
- templates/index.html — home page
- templates/home.html — prediction form and result
- requirements.txt — packages to install

The transcript stores a scaler and Ridge model separately. Saving one fitted pipeline is safer because the app cannot forget to scale before predicting.

In [ ]:
# Cell 2: recreate a fitted Ridge pipeline for this runnable lesson
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

df = pd.read_csv('Model Training Practicals/Algerian_forest_fires_cleaned_dataset.csv')
df['Classes'] = df['Classes'].astype(str).str.strip().str.lower().map({'not fire': 0, 'fire': 1})
feature_names = [name for name in df.columns if name not in ['day', 'month', 'year', 'FWI']]
X, y = df[feature_names], df['FWI']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_train, y_train)
print('Model input fields:', feature_names)

### What changed from model training?

The trained pipeline is the object a web app needs. It contains the fitted StandardScaler and Ridge model together. In the real project, load it once when the Flask application starts; do not retrain it for every request.

In [ ]:
# Cell 3: show the requirements.txt content
requirements = ['Flask', 'numpy', 'pandas', 'scikit-learn']
print('\n'.join(requirements))
print('\nInstall once with: pip install -r requirements.txt')

### Why requirements.txt?

It gives every computer and cloud service the list of packages the app needs. Update this file whenever the application imports a new package. Pin package versions for a real deployment after testing them.

In [ ]:
# Cell 4: Flask application code, kept runnable inside this notebook
from flask import Flask, request, render_template_string

app = Flask(__name__)

PAGE = '''
<h1>Algerian Forest Fire FWI Predictor</h1>
<form method='post'>
{% for name in feature_names %}
  <label>{{ name }} <input name='{{ name }}' required></label><br>
{% endfor %}
  <button type='submit'>Predict FWI</button>
</form>
{% if result is not none %}<h2>Predicted FWI: {{ result }}</h2>{% endif %}
{% if error %}<p style='color:crimson'>{{ error }}</p>{% endif %}
'''

@app.route('/', methods=['GET', 'POST'])
def predict_fwi():
    if request.method == 'GET':
        return render_template_string(PAGE, feature_names=feature_names, result=None, error=None)
    try:
        values = {name: float(request.form[name]) for name in feature_names}
        new_data = pd.DataFrame([values], columns=feature_names)
        result = round(float(ridge_pipeline.predict(new_data)[0]), 2)
        return render_template_string(PAGE, feature_names=feature_names, result=result, error=None)
    except (KeyError, ValueError):
        return render_template_string(PAGE, feature_names=feature_names, result=None, error='Enter a valid number in every field.'), 400

print('Flask app and prediction route created.')

### Read the route line by line

- The route slash is the page URL.
- GET displays the blank form.
- POST happens after the user submits the form.
- request.form reads the input fields by name.
- A one-row DataFrame preserves the feature names and order.
- The pipeline performs scaling and Ridge prediction in one call.
- The try/except block returns a useful message instead of crashing on invalid input.

In a folder-based project, replace render_template_string with render_template and put the HTML in templates/home.html. Flask looks in the templates folder by default.

In [ ]:
# Cell 5: test GET and POST without opening a browser
client = app.test_client()
get_response = client.get('/')
sample_form = {name: str(X_test.iloc[0][name]) for name in feature_names}
post_response = client.post('/', data=sample_form)
print('GET status:', get_response.status_code)
print('POST status:', post_response.status_code)
print('Prediction returned:', b'Predicted FWI' in post_response.data)

In [ ]:
# Cell 6: test the validation message with one invalid value
invalid_form = sample_form.copy()
invalid_form['Temperature'] = 'hot'
invalid_response = client.post('/', data=invalid_form)
print('Invalid-input status:', invalid_response.status_code)
print('Helpful error returned:', b'Enter a valid number' in invalid_response.data)

## Save and load the model in application.py

In the real app, save the pipeline once after training. Then application.py loads it during startup:

    import pickle
    with open('models/ridge_pipeline.pkl', 'rb') as file:
        ridge_pipeline = pickle.load(file)

Only load trusted pickle files. Pickles can execute unsafe code if they come from an unknown source.

In [ ]:
# Cell 7: show what Flask returns for one valid form submission
html_preview = post_response.data.decode('utf-8')
start = html_preview.find('<h2>')
end = html_preview.find('</h2>') + len('</h2>')
print(html_preview[start:end])

## Git and deployment checklist

1. Keep code, templates, requirements, and trusted model artifacts in the project folder.
2. Test the Flask route locally before deployment.
3. Add files to Git, commit with a clear message, and push to your GitHub repository.
4. The next deployment stage can use a service such as AWS Elastic Beanstalk or another supported cloud platform.
5. Never commit passwords, cloud keys, or other secrets. Use environment variables instead.

**One-line interview answer:** A Flask ML app collects form data with POST, validates it, applies the saved preprocessing and model, then renders the prediction back to the user.